<a href="https://colab.research.google.com/github/TrajanDS/SFT_Agent_POC/blob/main/secondary_qwin_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installs: Working on colab run time after 2026.04

In [ ]:
# 1. Remove conflicting packages
!pip uninstall -y torch torchvision torchaudio torchtext \
  triton bitsandbytes \
  transformers tokenizers accelerate peft trl datasets \
  numpy pandas pyarrow

In [ ]:
# 2. Install PyTorch CUDA 11.8 stack
# Moving from torch 2.2.0 -> 2.3.1 is safer on modern Colab / Python 3.12.
# PyTorch publishes a CUDA 11.8 wheel for torch 2.3.1.
!pip install --no-cache-dir \
  torch==2.3.1 \
  torchvision==0.18.1 \
  torchaudio==2.3.1 \
  --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# 3. Install Hugging Face + data stack
# bitsandbytes 0.45.4 avoids the old triton.ops import issue better than 0.43.x.
!pip install --no-cache-dir \
  numpy==1.26.4 \
  pandas==2.2.2 \
  pyarrow==15.0.2 \
  transformers==4.38.2 \
  tokenizers==0.15.2 \
  peft==0.9.0 \
  accelerate==0.27.2 \
  trl==0.8.1 \
  datasets==2.18.0 \
  bitsandbytes==0.45.4 \
  scipy==1.12.0 \
  fsspec==2024.2.0

# <font color = red> RESTART RUNTIME HERE

# Imports

In [ ]:
import torch
import tokenizers
import peft
import accelerate
import trl
from datasets import Dataset
import bitsandbytes as bnb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import google.generativeai as genai
from google.colab import userdata
from tqdm.notebook import tqdm
from google.colab import drive
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
    AutoModelForCausalLM,
    AutoTokenizer
)
from sklearn.model_selection import train_test_split
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)
from trl import SFTTrainer
import gc

In [ ]:
# Mount Google Drive
### (when prompted, approve all permissions and continue)
drive.mount('/content/drive')

# Fetch your API key from Colab Secrets
#GOOGLE_API_KEY = userdata.get('')
genai.configure(api_key='INPUT API KEY HERE')

# Initialize the Gemini model
gemini_model = genai.GenerativeModel('models/gemini-2.5-flash')

# Set output directory (local) path for images and on drive
drive_output_dir = "/content/drive/MyDrive/CFG_complaints/raw_data"

# Read in CFG complaints dataset using pyarrow directly, then convert to pandas

comp_df = pq.read_table(f"{drive_output_dir}/cfpb_complaints.parquet").to_pandas()

# Define the model name globally
qwen_model_name = "Qwen/Qwen1.5-4B-Chat"

# Load tokenizer globally and configure padding
tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
tokenizer.pad_token = tokenizer.eos_token # Qwen uses EOS as padding token by default
tokenizer.padding_side = "right" # Or "left" depending on preference for inference

## Fine-tuning Qwen3.5-4B with LoRA

To improve the classification performance of Qwen3.5-4B on our specific complaint dataset, we will fine-tune the model using Parameter-Efficient Fine-Tuning (PEFT) with LoRA (Low-Rank Adaptation). This technique allows us to adapt the model to our task by only training a small number of new parameters, significantly reducing computational cost and memory requirements.

### Steps:
1.  **Install necessary libraries**: `bitsandbytes` for 4-bit quantization, `peft` for LoRA, and `trl` for simplified supervised fine-tuning.
2.  **Prepare the dataset**: Format our `comp_df` into a conversational template suitable for instruction fine-tuning, mapping complaint narratives to product types.
3.  **Load the base model**: Load Qwen3.5-4B with 4-bit quantization to save memory.
4.  **Configure LoRA**: Define the LoRA parameters, such as rank and target modules.
5.  **Set up training arguments**: Define hyperparameters for training, like learning rate, batch size, and number of epochs.
6.  **Train the model**: Use `SFTTrainer` from `trl` to perform the fine-tuning.
7.  **Save and merge adapters**: Save the trained LoRA adapters and merge them into the base model.
8.  **Evaluate the fine-tuned model**: Test its classification performance on a separate, unseen subset of the data.

In [ ]:
# Filter out rows where 'complaint_what_happened' or 'product' is missing
cleaned_df = comp_df.dropna(subset=['complaint_what_happened', 'product']).copy()

# For demonstration, let's use a smaller subset of the data for fine-tuning
# Fine-tuning on the full dataset might take a very long time.
# We'll use 500 examples for training and 100 for evaluation.
if len(cleaned_df) > 600:
    # Stratified split to ensure product distribution is maintained
    train_df, eval_df = train_test_split(
        cleaned_df.sample(n=600, random_state=42),
        test_size=100,
        random_state=42,
        stratify=cleaned_df.sample(n=600, random_state=42)['product']
    )
else:
    train_df, eval_df = train_test_split(cleaned_df, test_size=0.2, random_state=42)


# Define the formatting function for the chat template
def format_chat_template(row):
    product_types = [
        'Checking or savings account',
        'Money transfer, virtual currency, or money service',
        'Vehicle loan or lease',
        'Credit card',
        'Payday loan, title loan, personal loan, or advance loan',
        'Credit reporting or other personal consumer reports',
        'Mortgage',
        'Student loan',
        'Debt collection',
        'Debt or credit management'
    ]
    messages = [
        {"role": "system", "content": "You are an expert financial complaint analyst. Your task is to classify the product type of the following consumer complaint. Based on the provided details, identify the most appropriate 'product' category from the predefined list. Respond with ONLY the product category string."},
        {"role": "user", "content": f"Complaint: {row['complaint_what_happened']}\nPredefined Product Categories: {product_types}"},
        {"role": "assistant", "content": row['product']}
    ]
    # Apply the tokenizer's chat template to format the conversation
    # Note: This requires the tokenizer to be loaded first, but we will define it here for clarity
    # The actual tokenization and formatting will happen within SFTTrainer.
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

# Convert DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df).map(format_chat_template, remove_columns=list(train_df.columns))
eval_dataset = Dataset.from_pandas(eval_df).map(format_chat_template, remove_columns=list(eval_df.columns))

print("Training data sample:")
print(train_dataset[0]['text'])
print("\nEvaluation data sample:")
print(eval_dataset[0]['text'])

In [ ]:
# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

# Load the base model with quantization
model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    quantization_config=quantization_config,
    device_map="auto"
)
model.config.use_cache = False
model.config.pretraining_tp = 1 # Recommended for Qwen

In [ ]:
# Prepare model for k-bit training (important for 4-bit fine-tuning)
model = prepare_model_for_kbit_training(model)

# Define LoRA configuration
lora_config = LoraConfig(
    r=16, # LoRA attention dimension
    lora_alpha=32, # Alpha parameter for LoRA scaling
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Target all linear layers in Qwen
    lora_dropout=0.05, # Dropout probability for LoRA layers
    bias="none", # No bias in LoRA layers
    task_type="CAUSAL_LM", # Causal Language Modeling for instruction tuning
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

print(model.print_trainable_parameters())

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./qwen_finetuned", # Output directory for checkpoints and logs
    num_train_epochs=3, # Number of training epochs
    per_device_train_batch_size=1, # Batch size per GPU/CPU for training
    gradient_accumulation_steps=4, # Number of updates steps to accumulate gradients
    gradient_checkpointing=True, # Enable gradient checkpointing for memory efficiency
    optim="paged_adamw_8bit", # Optimizer (8-bit AdamW for memory efficiency)
    logging_steps=10, # Log training metrics every N steps
    learning_rate=2e-4, # Learning rate
    fp16 = True,
    bf16=False, # Use bfloat16 if available
    tf32=False, # Use tf32 if available
    max_grad_norm=0.3, # Max gradient norm
    warmup_ratio=0.03, # Warmup ratio for learning rate scheduler
    lr_scheduler_type="constant", # Learning rate scheduler type
    report_to="none", # Disable reporting to W&B or other platforms
    # Evaluation arguments
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

In [ ]:
# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=512, # Maximum sequence length for inputs
    dataset_text_field="text", # Field in dataset containing the formatted text
    packing=True, # Pack multiple examples into one sequence for efficiency
)

# Start training
print("\nStarting Qwen fine-tuning...")
trainer.train()
print("\nFine-tuning complete!")

In [ ]:

# 1. Clear VRAM and local memory
if 'model' in locals(): del model
if 'trainer' in locals(): del trainer
gc.collect()
torch.cuda.empty_cache()

output_dir = "./qwen_finetuned_adapters"
offload_dir = "./offload_temp"

print("Loading base model and merging adapters... this may take a moment.")

# 2. Reload the base model with offloading enabled
base_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto",
    offload_folder=offload_dir,
    low_cpu_mem_usage=True
)

tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 3. Load the PEFT model with offload_folder and merge
# Passing offload_folder here fixes the ValueError during dispatch
finetuned_model = PeftModel.from_pretrained(
    base_model,
    output_dir,
    offload_folder=offload_dir
)

print("Merging adapters...")
finetuned_model = finetuned_model.merge_and_unload()
finetuned_model.eval()

print("Fine-tuned model loaded and adapters merged successfully.")

In [ ]:
# Update the classification function to use the fine-tuned model
def classify_product_with_finetuned_qwen(
    complaint_row,
    model,
    tokenizer,
    product_types = [
        'Checking or savings account',
        'Money transfer, virtual currency, or money service',
        'Vehicle loan or lease',
        'Credit card',
        'Payday loan, title loan, personal loan, or advance loan',
        'Credit reporting or other personal consumer reports',
        'Mortgage',
        'Student loan',
        'Debt collection',
        'Debt or credit management'
    ]
    ):
    messages = [
        {"role": "system", "content": "You are an expert financial complaint analyst. Your task is to classify the product type of the following consumer complaint. Based on the provided details, identify the most appropriate 'product' category from the predefined list. Respond with ONLY the product category string."},
        {"role": "user", "content": f"Complaint: {complaint_row['complaint_what_happened']}\nPredefined Product Categories: {product_types}"}
    ]

    # Tokenize the prompt
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    try:
        # Generate response from Qwen model
        generated_ids = model.generate(
            model_inputs.input_ids,
            max_new_tokens=50, # Limit the output length
            do_sample=False, # Use greedy decoding for deterministic classification
            temperature=0.0, # Set temperature to 0 for deterministic output
        )
        # Decode only the newly generated tokens
        input_len = model_inputs.input_ids.shape[1]
        response_text = tokenizer.decode(generated_ids[0, input_len:], skip_special_tokens=True).strip()

        # Post-process the response to get only the product category
        predicted_category = "Unknown"
        for p_type in product_types:
            if p_type.lower() in response_text.lower():
                predicted_category = p_type
                break
        # Fallback to the direct response if no predefined category is found and response is short
        if predicted_category == "Unknown" and len(response_text.split()) < 5: # Assuming category is short
             predicted_category = response_text # Take the model's direct output

        return predicted_category
    except Exception as e:
        print(f"Error classifying complaint ID {complaint_row['complaint_id']}: {e}")
        return "Error_Classification"

print("Running fine-tuned Qwen classification on the evaluation subset (first 10 rows)...")
# Apply the fine-tuned Qwen classification function to the evaluation set
finetuned_qwen_inf_df = eval_df.head(100).apply(
    lambda row: classify_product_with_finetuned_qwen(row, finetuned_model, tokenizer),
    axis=1
)

# Add the Qwen classifications to a new column in eval_df for comparison
eval_df = eval_df.head(10).copy() # Use the subset to match inference results
eval_df['finetuned_qwen_classified_product'] = finetuned_qwen_inf_df

# Display the value counts of the fine-tuned Qwen-classified products
print("\nFine-tuned Qwen Classified Product Value Counts:")
display(eval_df['finetuned_qwen_classified_product'].value_counts())

# Display a comparison of original vs. fine-tuned Qwen classified for a few rows
print("\nComparison of Original vs. Fine-tuned Qwen Classified Products (first 10 rows):")
display(eval_df[['product', 'finetuned_qwen_classified_product', 'complaint_what_happened']])

# Calculate accuracy on the evaluation subset
correct_finetuned_qwen_classifications = (eval_df['product'] == eval_df['finetuned_qwen_classified_product']).sum()
total_finetuned_qwen_classifications = len(eval_df)
accuracy_finetuned_qwen = correct_finetuned_qwen_classifications / total_finetuned_qwen_classifications
print(f"\nFine-tuned Qwen Classification Accuracy (vs. original 'product' column): {accuracy_finetuned_qwen:.2f}")

In [ ]:
finetuned_qwen_inf_df

In [ ]:
eval_df[['product', 'finetuned_qwen_classified_product']]

In [ ]:
eval_df['pred_wrong'] = eval_df.product == eval_df.finetuned_qwen_classified_product

In [ ]:
eval_df.pred_wrong.value_counts()